# Fases 1 a 3 - Corpus, limpieza y baseline SVM

El notebook usa la base reusable de `src/` y deja listo el baseline clasico sobre una muestra estratificada real.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    plt = None
    sns = None

RAIZ_PROYECTO = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.artefactos_modelos import guardar_modelo_svm
from src.corpus_inmuebles import construir_tabla_distribucion_clases, preparar_corpus_para_modelado
from src.evaluacion_modelos import (
    construir_matriz_confusion_tabla,
    construir_reporte_clasificacion,
    construir_tabla_metricas,
    evaluar_svm_con_validacion_cruzada,
)
from src.infraestructura_cpu import configurar_torch_cpu, relevar_hardware, sugerir_tamanio_muestra
from src.property_text_pipeline import (
    COLUMNA_OBJETIVO,
    COLUMNA_TEXTO_LIMPIO,
    TERMINOS_CLAVE,
    construir_auditoria_terminos,
    construir_ejemplos_limpieza,
    entrenar_modelo_base_svm,
)


In [ ]:
RUTA_DATOS = RAIZ_PROYECTO / 'data' / 'entrenamiento.csv'
SEMILLA = 42
TAMANIO_TEST = 0.2

resumen = relevar_hardware()
configurar_torch_cpu()
tamanio_muestra = sugerir_tamanio_muestra(resumen.memoria_disponible_gb)
print(f'Tamanio de muestra sugerido: {tamanio_muestra}')

df_muestra, df_entrenamiento, df_prueba = preparar_corpus_para_modelado(
    ruta_datos=RUTA_DATOS,
    tamanio_muestra=tamanio_muestra,
    tamanio_test=TAMANIO_TEST,
    semilla=SEMILLA,
)

print(f'Registros en la muestra: {len(df_muestra)}')
construir_tabla_distribucion_clases(df_muestra)


In [ ]:
ratio_vacios = (df_muestra[COLUMNA_TEXTO_LIMPIO].str.len() == 0).mean()
print(f'Proporcion de texto limpio vacio: {ratio_vacios:.4f}')

ejemplos = construir_ejemplos_limpieza(df_muestra)
auditoria = construir_auditoria_terminos(df_muestra, terminos_clave=TERMINOS_CLAVE)

display(ejemplos)
display(auditoria)


In [ ]:
modelo_svm = entrenar_modelo_base_svm(
    df_entrenamiento,
    df_entrenamiento[COLUMNA_OBJETIVO],
    columna_texto=COLUMNA_TEXTO_LIMPIO,
)
predicciones_svm = modelo_svm.predict(df_prueba[COLUMNA_TEXTO_LIMPIO])

print(construir_reporte_clasificacion(df_prueba[COLUMNA_OBJETIVO], predicciones_svm))
display(construir_tabla_metricas(df_prueba[COLUMNA_OBJETIVO], predicciones_svm))
display(evaluar_svm_con_validacion_cruzada(df_muestra))

matriz_confusion = construir_matriz_confusion_tabla(
    df_prueba[COLUMNA_OBJETIVO],
    predicciones_svm,
    etiquetas_ordenadas=['Departamento', 'Casa', 'PH'],
)
display(matriz_confusion)

if plt is not None and sns is not None:
    plt.figure(figsize=(7, 5))
    sns.heatmap(matriz_confusion, annot=True, fmt='d', cmap='Blues')
    plt.title('Matriz de confusion - SVM')
    plt.tight_layout()
    plt.show()


In [ ]:
ruta_modelo = guardar_modelo_svm(modelo_svm)
print(f'Modelo SVM guardado en: {ruta_modelo}')
